# Tests: `fastermodels.card` (source `nbs/02_card.ipynb`)

In [ ]:
from fastcore.test import *
from fastermodels.card import FORBIDDEN, check_card, render_card

In [ ]:
_meta = dict(
    name='test-resnet18-imagenette', base_model='torchvision/resnet18', license='bsd-3-clause',
    datasets=['frgfm/imagenette'], tags=['fasterai', 'pruning'],
    scope_line='Imagenette, n=3925, Wilson half-width about 1 pt; pipeline evidence, not a published claim.',
    recipe={'prune': 'ratio 0.3, local, round_to 8', 'recovery': '3 epochs'},
    reference={'name': 'resnet18 fine-tuned on Imagenette', 'k': 3700, 'n': 3925},
    rows=[dict(artifact='pruned FP32', file='model.safetensors', params=8_900_000, bytes=35_600_000,
               k=3680, n=3925, delta=-0.51, lo=-1.2, hi=0.2, p_mcnemar=0.12, agreement=1.0,
               agreement_kind='same-precision')],
    latency=None,
    provenance={'fasterai': '0.4.0', 'fastermodels': '0.1.0', 'torch': '2.9.1', 'measured_on': '2026-09-11'})

_card = render_card(_meta)

# a complete card says nothing a reader has to take on trust
test_eq(check_card(_card), [])

# the front matter the Hub reads
assert _card.startswith('---\n')
assert 'library_name: fastermodels' in _card
assert 'license: bsd-3-clause' in _card
assert 'base_model: torchvision/resnet18' in _card
assert '  - frgfm/imagenette' in _card

# the numbers, with the count they were measured on and their interval
assert '3700/3925' in _card and '3680/3925' in _card
assert 'Wilson 95 %' in _card
assert '-0.51' in _card and '[-1.20, +0.20]' in _card
assert 'same-precision' in _card

# a latency that was not measured says so, it never reads as a zero
assert 'non mesurée' in _card
assert '0.00 ms' not in _card

In [ ]:
# a measured latency is written with the device, the runtime, the precision and the batch
_measured = render_card({**_meta, 'latency': [dict(device='Jetson Orin NX', runtime='tensorrt', precision='fp16',
                                                   batch=1, median_ms=0.507, n_runs=100)]})
assert 'non mesurée' not in _measured
assert 'Jetson Orin NX' in _measured and 'tensorrt' in _measured and '0.507' in _measured
test_eq(check_card(_measured), [])

In [ ]:
# every forbidden phrase is reported, once, wherever it is injected
for _p in FORBIDDEN:
    test_eq(check_card(_card + f'\nThe artifact is {_p} on this line.\n'), [_p])
    test_eq(check_card(_card + f'\nThe artifact is {_p.upper()} on this line.\n'), [_p])

# whole words only: a phrase inside a longer word is not a claim
test_eq(check_card('the nanometre scale'), [])
test_eq(check_card('todolist'), [])

In [ ]:
# a speedup with neither device nor runtime on its line is reported
_flagged = check_card('the model is 2.3x faster')
test_eq(len(_flagged), 1)
assert '2.3x' in _flagged[0]
test_eq(len(check_card('the model is 2.3× faster')), 1)
test_eq(len(check_card('3x faster')), 1)

# the same claim with the device and the runtime it was measured on is not
test_eq(check_card('the model is 2.3x on CPU with onnxruntime'), [])
test_eq(check_card('2.3x vs the FP32 engine on a Jetson Orin NX with tensorrt, batch 1'), [])

# a number that is not a speedup is not a claim
test_eq(check_card('a 1x1 convolution'), [])
test_eq(check_card('the matrix is 3x4'), [])